# 07 · Label-smoothing sweep

**Purpose.** Train the label-smoothing variants of C2 and C3 (ε ∈ {0.05, 0.1, 0.2, 0.3, 0.5}; ε = 0 reuses the base model), 16 shadows each, attack them and trace the frontier.

**Inputs.** Feature cache, split file, base-config models (notebooks 02, 04).

**Outputs.** Configs `C2_ls*`, `C3_ls*`, their models and attack runs, `sweep_points` (x = ε), figure F4.

**Syllabus topics.** Discriminant functions with smoothed targets; PCA.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from mia.training import Workspace, register_sweep_points, run_jobs

ws = Workspace.open()
exp = ws.exp
ls_ids = [exp.ls_config(b, e).config_id for b in exp.ls_bases for e in exp.ls_epsilons if e > 0]
jobs = []
for cid in ls_ids:
    jobs.append((cid, 0, None))
    jobs += [(cid, 0, k) for k in range(exp.classical_k)]
print(len(jobs), "training runs for", ls_ids)
summary = run_jobs(ws, jobs)

In [ ]:
from mia import db
from mia.attacks import run_all_attacks

for cid in ls_ids:
    for (mid,) in ws.conn.execute("SELECT model_id FROM models WHERE config_id = ? AND role = 'target'", (cid,)):
        print(f"{mid:<26} {len(run_all_attacks(ws.conn, exp, mid, ws.labels))} new attack run(s)")
register_sweep_points(ws.conn, exp)
ls = db.sweep_df(ws.conn, "label_smoothing")
ls[["base_config_id", "x_value", "model_id", "train_acc", "test_acc", "gap_acc", "attack", "auc", "tpr_at_1pct_fpr", "tpr_at_01pct_fpr"]].round(4)

In [ ]:
import matplotlib.pyplot as plt

from mia import plots

plots.apply_style()
fig = plots.f4_label_smoothing(ls)
print("saved:", *plots.save_figure(fig, "F4_label_smoothing"))
plt.show()

In [ ]:
for base, g in ls.groupby("base_config_id"):
    a0, rest = g[g["x_value"] == 0]["auc"].mean(), g[g["x_value"] > 0]["auc"].mean()
    print(f"{base}: AUC at eps=0 {a0:.3f}; mean AUC over eps>0 {rest:.3f} -> label smoothing "
          f"{'REDUCED' if rest < a0 else 'INCREASED'} leakage")
print(f"Training: trained {summary['trained']}, skipped {summary['skipped']}, failed {summary['failed']}")

## Results
The printout states, per base config, whether label smoothing reduced or increased leakage relative to ε = 0; F4 shows the accuracy cost of each ε.